# 02 - Exploratory Data Analysis (EDA)

**Objective:** Create 30+ visualizations and answer 15+ business questions.

## Imports Libraries & Load Data 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


In [2]:
# Load cleaned data
try:
    orders = pd.read_csv('../data/cleaned/orders_cleaned.csv')
    customers = pd.read_csv('../data/cleaned/customers_cleaned.csv')
    restaurants = pd.read_csv('../data/cleaned/restaurants_cleaned.csv')
    menu = pd.read_csv('../data/cleaned/menu_cleaned.csv')
    delivery_partners = pd.read_csv('../data/cleaned/delivery_partners_cleaned.csv')
    customer_feedback = pd.read_csv('../data/cleaned/customer_feedback_cleaned.csv')
    
    print('✅ All datasets loaded successfully')
    print(f'\nOrders shape: {orders.shape}')
    print(f'Customers shape: {customers.shape}')
    print(f'Restaurants shape: {restaurants.shape}')
except Exception as e:
    print(f'Error loading data: {e}')
    print('Make sure you ran notebook 01_data_cleaning.ipynb first')

Error loading data: [Errno 2] No such file or directory: '../data/cleaned/orders_cleaned.csv'
Make sure you ran notebook 01_data_cleaning.ipynb first


## Q1: Revenue Analysis

In [3]:
# Top restaurants by revenue
# Top cuisines by revenue# Top 10 restaurants by revenue
restaurant_revenue = orders.groupby('restaurantid')['finalamount'].sum().sort_values(ascending=False).head(10)

# Merge with restaurant names
top_restaurants = restaurants[restaurants['restaurantid'].isin(restaurant_revenue.index)][['restaurantid', 'restaurantname']]
restaurant_revenue_df = pd.DataFrame({
    'Revenue': restaurant_revenue.values
}, index=top_restaurants.set_index('restaurantid').loc[restaurant_revenue.index, 'restaurantname'].values)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
restaurant_revenue_df.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 10 Restaurants by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Revenue (₹)', fontsize=12)
ax.set_ylabel('Restaurant Name', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Top Restaurant: {restaurant_revenue_df.index[0]} - ₹{restaurant_revenue_df.iloc[0, 0]:,.0f}')
# Revenue trends over time

NameError: name 'orders' is not defined

In [ ]:
# Revenue by Cuisine
merged_data = orders.merge(restaurants[['restaurantid', 'cuisine']], on='restaurantid')
cuisine_revenue = merged_data.groupby('cuisine')['finalamount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
cuisine_revenue.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Revenue by Cuisine Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Cuisine', fontsize=12)
ax.set_ylabel('Total Revenue (₹)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\nTop Cuisine: {cuisine_revenue.index[0]} - ₹{cuisine_revenue.iloc[0]:,.0f}')
print(f'Total Cuisines: {len(cuisine_revenue)}')

In [ ]:
# Monthly Revenue Trend
orders['orderdate'] = pd.to_datetime(orders['orderdate'], errors='coerce')
orders['month'] = orders['orderdate'].dt.to_period('M')
monthly_revenue = orders.groupby('month')['finalamount'].sum()

fig, ax = plt.subplots(figsize=(12, 6))
monthly_revenue.plot(ax=ax, color='green', linewidth=2, marker='o')
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Revenue (₹)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Average Monthly Revenue: ₹{monthly_revenue.mean():,.0f}')
print(f'Highest Month: {monthly_revenue.idxmax()} - ₹{monthly_revenue.max():,.0f}')

In [ ]:
# Revenue Distribution (Pie Chart)
city_revenue = merged_data.groupby('city')['finalamount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.Set3(range(len(city_revenue)))
ax.pie(city_revenue, labels=city_revenue.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Revenue Distribution by City', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nRevenue by City:')
for city, revenue in city_revenue.items():
    print(f'  {city}: ₹{revenue:,.0f} ({revenue/city_revenue.sum()*100:.1f}%)')

## Q2: Delivery Performance

In [ ]:
# Delivery Time Distribution
fig, ax = plt.subplots(figsize=(12, 6))
orders['deliverytimeminutes'].hist(bins=50, ax=ax, color='skyblue', edgecolor='black')
ax.axvline(orders['deliverytimeminutes'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {orders['deliverytimeminutes'].mean():.1f} min")
ax.axvline(orders['deliverytimeminutes'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {orders['deliverytimeminutes'].median():.1f} min")
ax.set_title('Delivery Time Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Delivery Time (Minutes)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean Delivery Time: {orders["deliverytimeminutes"].mean():.2f} minutes')
print(f'Median Delivery Time: {orders["deliverytimeminutes"].median():.2f} minutes')
print(f'Std Dev: {orders["deliverytimeminutes"].std():.2f} minutes')
print(f'Min: {orders["deliverytimeminutes"].min():.0f} minutes')
print(f'Max: {orders["deliverytimeminutes"].max():.0f} minutes')

In [ ]:
# Delivery Time by City (Box Plot)
city_delivery = merged_data[['city', 'deliverytimeminutes']]

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=city_delivery, x='city', y='deliverytimeminutes', ax=ax, palette='Set2')
ax.set_title('Delivery Time Distribution by City', fontsize=14, fontweight='bold')
ax.set_xlabel('City', fontsize=12)
ax.set_ylabel('Delivery Time (Minutes)', fontsize=12)
plt.tight_layout()
plt.show()

print('\nAverage Delivery Time by City:')
city_avg = city_delivery.groupby('city')['deliverytimeminutes'].mean().sort_values()
for city, time in city_avg.items():
    print(f'  {city}: {time:.2f} minutes')

In [ ]:
# Peak Order Hours
orders['hour'] = pd.to_datetime(orders['ordertime'], format='%H:%M:%S', errors='coerce').dt.hour
hourly_orders = orders.groupby('hour').size()

fig, ax = plt.subplots(figsize=(12, 6))
hourly_orders.plot(ax=ax, color='purple', linewidth=2, marker='o')
ax.set_title('Order Volume by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Number of Orders', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

peak_hour = hourly_orders.idxmax()
print(f'\nPeak Order Hour: {peak_hour}:00 with {hourly_orders[peak_hour]} orders')

## Q3: Customer Behavior Analysis

In [ ]:
# Customer Order Frequency
customer_orders = orders.groupby('customerid').size()

fig, ax = plt.subplots(figsize=(12, 6))
customer_orders.value_counts().sort_index().plot(kind='bar', ax=ax, color='teal')
ax.set_title('Customer Order Frequency Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Orders', fontsize=12)
ax.set_ylabel('Number of Customers', fontsize=12)
plt.tight_layout()
plt.show()

repeat_customers = (customer_orders > 1).sum()
one_time = (customer_orders == 1).sum()
print(f'\nRepeat Customers: {repeat_customers} ({repeat_customers/len(customer_orders)*100:.1f}%)')
print(f'One-time Customers: {one_time} ({one_time/len(customer_orders)*100:.1f}%)')
print(f'Average Orders per Customer: {customer_orders.mean():.2f}')

In [ ]:
# Basket Value Distribution
fig, ax = plt.subplots(figsize=(12, 6))
orders['finalamount'].hist(bins=50, ax=ax, color='orange', edgecolor='black')
ax.axvline(orders['finalamount'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: ₹{orders['finalamount'].mean():.0f}")
ax.set_title('Order Value Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Order Value (₹)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nAverage Order Value: ₹{orders["finalamount"].mean():.2f}')
print(f'Median Order Value: ₹{orders["finalamount"].median():.2f}')
print(f'Min Order Value: ₹{orders["finalamount"].min():.2f}')
print(f'Max Order Value: ₹{orders["finalamount"].max():.2f}')

In [ ]:
# Customer Lifetime Value
customer_clv = orders.groupby('customerid')['finalamount'].sum()

fig, ax = plt.subplots(figsize=(12, 6))
customer_clv.hist(bins=50, ax=ax, color='lightblue', edgecolor='black')
ax.set_title('Customer Lifetime Value Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Spending (₹)', fontsize=12)
ax.set_ylabel('Number of Customers', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nAverage CLV: ₹{customer_clv.mean():.2f}')
print(f'Median CLV: ₹{customer_clv.median():.2f}')
print(f'Top Customer CLV: ₹{customer_clv.max():.2f}')

## Q4: Correlation Analysis 

In [ ]:
# Correlation Heatmap
numeric_cols = orders.select_dtypes(include=[np.number]).columns
correlation_matrix = orders[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Matrix of Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Delivery Time vs Order Value
fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(orders['finalamount'], orders['deliverytimeminutes'], alpha=0.5, s=20, color='navy')
z = np.polyfit(orders['finalamount'].dropna(), orders[orders['finalamount'].notna()]['deliverytimeminutes'], 1)
p = np.poly1d(z)
ax.plot(orders['finalamount'].sort_values(), p(orders['finalamount'].sort_values()), "r--", linewidth=2, label='Trend')
ax.set_title('Delivery Time vs Order Value', fontsize=14, fontweight='bold')
ax.set_xlabel('Order Value (₹)', fontsize=12)
ax.set_ylabel('Delivery Time (Minutes)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

corr = orders['finalamount'].corr(orders['deliverytimeminutes'])
print(f'Correlation between Order Value and Delivery Time: {corr:.3f}')

## Q5: Payment Method Analysis

In [ ]:
# Payment Method Breakdown
payment_counts = orders['paymentmethod'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Count
payment_counts.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Order Count by Payment Method', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Orders')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# Revenue
payment_revenue = orders.groupby('paymentmethod')['finalamount'].sum().sort_values(ascending=False)
payment_revenue.plot(kind='bar', ax=ax2, color='coral')
ax2.set_title('Revenue by Payment Method', fontsize=12, fontweight='bold')
ax2.set_ylabel('Total Revenue (₹)')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print('\nPayment Method Distribution:')
for method, count in payment_counts.items():
    pct = count / payment_counts.sum() * 100
    print(f'  {method}: {count} orders ({pct:.1f}%)')

## Q6: Rating Analysis

In [ ]:
# Customer Rating Distribution
fig, ax = plt.subplots(figsize=(10, 6))
customer_feedback['customerrating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='yellowgreen')
ax.set_title('Customer Rating Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Rating (1-5 stars)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

avg_rating = customer_feedback['customerrating'].mean()
print(f'\nAverage Customer Rating: {avg_rating:.2f} / 5.0')
print(f'Most Common Rating: {customer_feedback["customerrating"].mode()[0]}')

In [ ]:
# Restaurant Ratings
fig, ax = plt.subplots(figsize=(12, 6))
restaurants['rating'].hist(bins=20, ax=ax, color='lightcoral', edgecolor='black')
ax.axvline(restaurants['rating'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {restaurants['rating'].mean():.2f}")
ax.set_title('Restaurant Rating Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Rating (1-5 stars)', fontsize=12)
ax.set_ylabel('Number of Restaurants', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nAverage Restaurant Rating: {restaurants["rating"].mean():.2f}')
print(f'Top Rated Restaurant: {restaurants[restaurants["rating"].idxmax()]["restaurantname"]} ({restaurants["rating"].max():.1f})')

## Summary Statistics

In [ ]:
print('='*70)
print('ZOMATO BUSINESS INTELLIGENCE - EDA SUMMARY')
print('='*70)

print('\n📊 REVENUE METRICS')
print(f'  Total Revenue: ₹{orders["finalamount"].sum():,.0f}')
print(f'  Average Order Value: ₹{orders["finalamount"].mean():.2f}')
print(f'  Highest Single Order: ₹{orders["finalamount"].max():.2f}')

print('\n🍽️ RESTAURANT METRICS')
print(f'  Total Restaurants: {len(restaurants)}')
print(f'  Average Rating: {restaurants["rating"].mean():.2f} / 5.0')
print(f'  Total Cuisines: {restaurants["cuisine"].nunique()}')

print('\n CUSTOMER METRICS')
print(f'  Total Customers: {len(customers)}')
print(f'  Total Orders: {len(orders)}')
print(f'  Repeat Customer Rate: {(customer_orders > 1).sum() / len(customer_orders) * 100:.1f}%')
print(f'  Average Orders per Customer: {customer_orders.mean():.2f}')

print('\n DELIVERY METRICS')
print(f'  Average Delivery Time: {orders["deliverytimeminutes"].mean():.2f} minutes')
print(f'  Median Delivery Time: {orders["deliverytimeminutes"].median():.2f} minutes')
print(f'  Fastest Delivery: {orders["deliverytimeminutes"].min():.0f} minutes')
print(f'  Slowest Delivery: {orders["deliverytimeminutes"].max():.0f} minutes')

print('\n RATING METRICS')
print(f'  Average Customer Rating: {customer_feedback["customerrating"].mean():.2f} / 5.0')
print(f'  Average Delivery Rating: {customer_feedback["deliveryrating"].mean():.2f} / 5.0')
print(f'  Average Food Rating: {customer_feedback["foodrating"].mean():.2f} / 5.0')

print('\n' + '='*70)